# 실습2: RNN (초급)
간단한 시퀀스 분류 예제를 통해 RNN 입력/출력 구조와 학습 흐름을 체험합니다. 작은 합성 데이터로 빠르게 실행해 보세요.

## 사전 준비
- 필요 패키지: Python, PyTorch, numpy, matplotlib
- 설치 예: pip install torch numpy matplotlib

In [ ]:
# 필수 라이브러리 불러오기
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 데이터 생성(합성 시퀀스)
- 두 클래스: (1) 잡음이 섞인 사인파, (2) 잡음이 섞인 사각파
- 시퀀스 길이: 50, 각 시퀀스는 1차원 값의 시간열

In [ ]:
# 합성 데이터 생성
def gen_sine(seq_len, num):
    x = np.linspace(0, 2*np.pi, seq_len)
    data = []
    for _ in range(num):
        phase = np.random.uniform(0, 2*np.pi)
        amp = np.random.uniform(0.8, 1.2)
        noise = np.random.normal(0, 0.1, size=seq_len)
        data.append(amp * np.sin(x + phase) + noise)
    return np.array(data)

def gen_square(seq_len, num):
    x = np.linspace(0, 2*np.pi, seq_len)
    data = []
    for _ in range(num):
        phase = np.random.uniform(0, 2*np.pi)
        base = np.sign(np.sin(x + phase))
        noise = np.random.normal(0, 0.1, size=seq_len)
        data.append(base + noise)
    return np.array(data)

seq_len = 50
n_per_class = 1000
sine = gen_sine(seq_len, n_per_class)
square = gen_square(seq_len, n_per_class)

X = np.concatenate([sine, square], axis=0)  # (2n, seq_len)
y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)], axis=0)

# torch 텐서 형태: (batch, seq_len, input_size=1)
X = torch.tensor(X, dtype=torch.float32).unsqueeze(-1)
y = torch.tensor(y, dtype=torch.long)

# 데이터로더
dataset = TensorDataset(X, y)
train_size = int(len(dataset) * 0.8)
train_set, test_set = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size])
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256)

print('Train samples:', len(train_set), 'Test samples:', len(test_set))

In [ ]:
# 데이터 시각화: 몇 개 샘플 확인
fig, axes = plt.subplots(2,2,figsize=(8,6))
for i, ax in enumerate(axes.flatten()):
    ax.plot(X[i].squeeze().numpy())
    ax.set_title(f'label={y[i].item()}')
plt.tight_layout()
plt.show()

## 모델 정의 (간단한 RNN)
- RNN 셀을 사용하여 시퀀스를 처리합니다.
- 마지막 타임스텝의 출력을 분류기로 사용합니다.

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=1, num_classes=2):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, _ = self.rnn(x)  # out: (batch, seq_len, hidden)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

model = SimpleRNN().to(device)
print(model)

In [ ]:
# 학습 (간단히 몇 epoch)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 8

for epoch in range(1, epochs+1):
    model.train()
    loss_sum = 0.0
    correct = 0
    total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * xb.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
    print(f'Epoch {epoch}/{epochs}  loss={loss_sum/total:.4f}  acc={correct/total:.4f}')

In [ ]:
# 평가
model.eval()
total = 0
correct = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        preds = out.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)
print('Test accuracy:', correct/total)

In [ ]:
# 예측 시각화: 일부 샘플과 예측 확인
xb, yb = next(iter(test_loader))
xb, yb = xb.to(device), yb.to(device)
with torch.no_grad():
    out = model(xb)
preds = out.argmax(dim=1).cpu()
xb = xb.cpu()
fig, axes = plt.subplots(2,5,figsize=(12,5))
for i, ax in enumerate(axes.flatten()):
    ax.plot(xb[i].squeeze().numpy())
    ax.set_title(f'GT:{yb[i].item()}  Pred:{preds[i].item()}')
plt.tight_layout()
plt.show()

실습 과제(쉬운 단계):
- hidden_size를 16 또는 64로 바꿔 성능 변화를 확인하세요.
- RNN을 LSTM으로 바꿔서 어떤 차이가 나는지 비교해 보세요.
- 노이즈 수준을 바꿔 모델 학습 안정성을 관찰하세요.

목표: RNN의 동작 방식을 간단히 이해하고, 하이퍼파라미터가 성능에 미치는 영향을 체감하는 것입니다.